# Olist Brazilian E-Commerce Analytics

Python / Pandas analysis of the Olist Brazilian E-Commerce dataset.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## Dataset path



## Load the datasets



In [ ]:
df_orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
df_order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
df_customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
df_products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
df_category_translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")
df_reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")


In [ ]:
print("Orders:", df_orders.shape)
print("Order items:", df_order_items.shape)
print("Customers:", df_customers.shape)
print("Products:", df_products.shape)
print("Reviews:", df_reviews.shape)


## Initial data-quality checks



In [ ]:
print("Duplicate customer_id:", df_customers["customer_id"].duplicated().sum())
print("Duplicate customer_unique_id:", df_customers["customer_unique_id"].duplicated().sum())
print("Duplicate order_id in orders:", df_orders["order_id"].duplicated().sum())
print("Duplicate order_id + order_item_id:", df_order_items.duplicated(["order_id", "order_item_id"]).sum())


## Prepare order timestamps



In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:


## Delivery analysis

### Business Question 3: Does delivery time relate to customer review scores?



In [ ]:
df_orders["actual_delivery_time"] = (
    df_orders["order_delivered_customer_date"]
    - df_orders["order_purchase_timestamp"]
)

df_orders["delivery_days"] = (
    df_orders["actual_delivery_time"].dt.total_seconds() / 86400
).round(2)



In [ ]:
quantiles = df_orders["delivery_days"].quantile([0.25, 0.50, 0.75, 0.90, 0.95])
iqr = quantiles.loc[0.75] - quantiles.loc[0.25]
upper_boundary = quantiles.loc[0.75] + 1.5 * iqr

print("Q1:", round(quantiles.loc[0.25], 2))
print("Median:", round(quantiles.loc[0.50], 2))
print("Q3:", round(quantiles.loc[0.75], 2))
print("P90:", round(quantiles.loc[0.90], 2))
print("P95:", round(quantiles.loc[0.95], 2))
print("IQR:", round(iqr, 2))
print("Upper IQR boundary:", round(upper_boundary, 3))


In [ ]:
df_orders["status_delivery_issue"] = np.where(
    (df_orders["order_status"] == "canceled")
    & df_orders["order_delivered_customer_date"].notna(),
    "Yes",
    "No"
)

print(df_orders["order_status"].value_counts())


### Delivered-order analysis



In [ ]:
df_delivered = df_orders[df_orders["order_status"] == "delivered"].copy()
df_delivered_analysis = df_delivered[df_delivered["delivery_days"].notna()].copy()

print("Delivered orders:", len(df_delivered))
print("Analyzed delivered orders:", len(df_delivered_analysis))
print("Mean delivery days:", round(df_delivered_analysis["delivery_days"].mean(), 2))


### Delivery time by customer state



In [ ]:
df_delivery_state = df_delivered_analysis.merge(
    df_customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

state_delivery = (
    df_delivery_state
    .groupby("customer_state")
    .agg(
        avg_delivery_days=("delivery_days", "mean"),
        median_delivery_days=("delivery_days", "median"),
        order_count=("order_id", "nunique")
    )
    .sort_values("avg_delivery_days", ascending=False)
)



### Review-score preparation



In [ ]:
review_score_by_order = (
    df_reviews
    .groupby("order_id")["review_score"]
    .mean()
    .rename("review_score")
)

df_delivery_reviews = (
    df_delivered_analysis[["order_id", "delivery_days"]]
    .merge(review_score_by_order, on="order_id", how="left")
    .dropna(subset=["review_score"])
)



In [ ]:
overall_correlation = df_delivery_reviews["delivery_days"].corr(
    df_delivery_reviews["review_score"]
)

normal_delivery_reviews = df_delivery_reviews[
    df_delivery_reviews["delivery_days"] <= upper_boundary
].copy()

sensitivity_correlation = normal_delivery_reviews["delivery_days"].corr(
    normal_delivery_reviews["review_score"]
)

print("Overall correlation:", round(overall_correlation, 2))


In [ ]:
normal_delivery_reviews["delivery_bucket"] = pd.cut(
    normal_delivery_reviews["delivery_days"],
    bins=[0, 5, 10, 15, 20, 25, upper_boundary],
    labels=["0-5", "5-10", "10-15", "15-20", "20-25", "25-29.145"],
    include_lowest=True
)

review_by_bucket = (
    normal_delivery_reviews
    .groupby("delivery_bucket", observed=False)
    .agg(
        average_review_score=("review_score", "mean"),
        order_count=("order_id", "nunique")
    )
)



## Question 1: Product category performance



In [ ]:
df_sales_over_time = (
    df_order_items
    .merge(df_products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(
        df_category_translation,
        on="product_category_name",
        how="left"
    )
)

category_sales = (
    df_sales_over_time
    .groupby("product_category_name_english", dropna=False)
    .agg(
        total_sales=("price", "sum"),
        items_sold=("order_item_id", "count"),
        avg_item_price=("price", "mean"),
        orders=("order_id", "nunique")
    )
)

category_sales["avg_sales_per_order"] = (
    category_sales["total_sales"] / category_sales["orders"]
)

category_sales = category_sales.sort_values("total_sales", ascending=False)


## Question 2: State sales performance



In [ ]:
df_state_sales = (
    df_order_items[["order_id", "price"]]
    .merge(df_orders[["order_id", "customer_id"]], on="order_id", how="left")
    .merge(df_customers[["customer_id", "customer_state"]], on="customer_id", how="left")
)

state_sales = (
    df_state_sales
    .groupby("customer_state")
    .agg(
        total_sales=("price", "sum"),
        orders=("order_id", "nunique")
    )
)

state_sales["avg_sales_per_order"] = (
    state_sales["total_sales"] / state_sales["orders"]
)



## Question 4: Sales over time



In [ ]:
sales_transactions = df_order_items.merge(
    df_orders[["order_id", "order_purchase_timestamp"]],
    on="order_id",
    how="left"
)

sales_transactions["purchase_month"] = (
    sales_transactions["order_purchase_timestamp"].dt.to_period("M")
)

monthly_sales = (
    sales_transactions
    .groupby("purchase_month")["price"]
    .sum()
    .sort_index()
)



In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(
    x=monthly_sales.index.to_timestamp(),
    y=monthly_sales.values,
    marker="o"
)
plt.xlabel("Month")
plt.ylabel("Merchandise Sales")
plt.title("Monthly Merchandise Sales")
plt.ticklabel_format(style="plain", axis="y")
plt.xticks(rotation=45)
plt.tight_layout()


## Question 5: New vs repeated orders



In [ ]:
orders_customers = (
    df_orders[["order_id", "customer_id", "order_purchase_timestamp"]]
    .merge(
        df_customers[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left"
    )
)

orders_customers = orders_customers.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
).copy()

orders_customers["purchase_rank"] = (
    orders_customers.groupby("customer_unique_id").cumcount() + 1
)

orders_customers["order_type"] = np.where(
    orders_customers["purchase_rank"] == 1,
    "New",
    "Repeated"
)

orders_customers["purchase_month"] = (
    orders_customers["order_purchase_timestamp"].dt.to_period("M")


In [ ]:
new_orders = (
    orders_customers[orders_customers["order_type"] == "New"]
    .groupby("purchase_month")["order_id"]
    .nunique()
)

monthly_orders = (
    orders_customers
    .groupby("purchase_month")["order_id"]
    .nunique()
)

repeat_orders = monthly_orders - new_orders

new_repeat_monthly = pd.DataFrame({
    "total_orders": monthly_orders,
    "new_orders": new_orders,
    "repeat_orders": repeat_orders
}).fillna(0)



## Question 6: Sales and order-volume changes



In [ ]:
sales_orders_monthwise = pd.concat(
    [monthly_sales.rename("sales"), monthly_orders.rename("orders")],
    axis=1
)

sales_orders_monthwise["average_sales_per_order"] = (
    sales_orders_monthwise["sales"] / sales_orders_monthwise["orders"]
)

comparison_months = [
    pd.Period("2017-08"), pd.Period("2017-09"),
    pd.Period("2018-03"), pd.Period("2018-04"),
    pd.Period("2018-07"), pd.Period("2018-08")
]

sales_orders_monthwise.loc[
    comparison_months,
    ["sales", "orders", "average_sales_per_order"]


### Category-mix check



In [ ]:
category_mix = (
    df_sales_over_time[
        df_sales_over_time["purchase_month"].isin(comparison_months)
    ]
    .groupby(
        ["purchase_month", "product_category_name_english"]
    )["price"]
    .sum()
    .reset_index()
)

category_mix.sort_values(
    ["purchase_month", "price"],
    ascending=[True, False]


## Key analytical outputs

